# 21_ensemble — 상위 4 모델 선정 + 앙상블 (신규)

**한 줄 요약:** 여러 알고리즘을 **검증셋(val)으로 비교**해 좋은 것 **4개를 골라** 하나의 **앙상블(soft voting)** 로 묶고, 최종 성능은 **시험셋(test)으로 딱 한 번** 잰다.
**후보 5종:** RandomForest·ExtraTrees·LightGBM·XGBoost·LogisticRegression.
**두 갈래:** (A) **전체 특징**(지문+2D+3D) = 최고 성능 평가용, (B) **fingerprint 전용** = 대형 라이브러리 스크리닝 배포용(700k엔 descriptor 계산이 비현실적이라 지문만).
**용어:** 앙상블=여러 모델의 예측을 **평균**해 더 안정적으로 / soft voting=확률 평균.
**큰 흐름:** ① 준비 → ② 데이터·split → ③ 후보 모델·지표 → ④ 비교·선정·앙상블·평가 → ⑤ 배포모델 저장

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
여러 분류기와 앙상블(`VotingClassifier`), 전처리(`Pipeline`), 지표를 가져온다.

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())
import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix, matthews_corrcoef)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

🔎 **코드 뜯어보기 (준비)**
- `RandomForestClassifier, ExtraTreesClassifier` : 트리 앙상블 모델. `VotingClassifier` : 여러 모델을 묶어 **투표/평균**하는 앙상블.
- `from lightgbm import LGBMClassifier` / `from xgboost import XGBClassifier` : 부스팅 트리 모델(강력).
- `Pipeline`(전처리+모델 묶음), `SimpleImputer`(결측 채움), `StandardScaler`(스케일 정규화, 선형모델용).

### 셀 1 — 데이터 + split 불러오기
v2 최종데이터에 20a의 split(train/val/test)을 붙이고, 지문 열과 전체 특징 열을 구분한다.

In [ ]:
# 최종데이터(v2) + split 불러와 train/val/test 로 나누기
DATA = "data/HSD17B13_final_training_1to1_v2.csv"
SPLIT = "data/HSD17B13_split_1to1.csv"
df = pd.read_csv(DATA)
sp = pd.read_csv(SPLIT)
df = df.merge(sp, on="canonical_smiles", how="left")     # split(train/val/test) 열 붙이기

fp_prefixes = ("ecfp4_", "rdkit_", "atompair_", "topotorsion_", "maccs_", "avalon_")
fp_cols = [c for c in df.columns if c.startswith(fp_prefixes)]              # 지문 열(스크리닝용)
all_cols = [c for c in df.columns if c not in ("canonical_smiles", "potency", "split")]  # 전체 특징
print(f"전체 특징 {len(all_cols)} | 그중 fingerprint {len(fp_cols)} | 나머지(2D+3D desc) {len(all_cols)-len(fp_cols)}")

def xy(cols, part):
    m = df["split"] == part
    return df.loc[m, cols].to_numpy(), df.loc[m, "potency"].to_numpy()
for p in ("train", "val", "test"):
    print(f"  {p}: {(df['split']==p).sum()}개")

🔎 **코드 뜯어보기 (셀 1)**
- `df.merge(sp, on="canonical_smiles")` : 분자별 **split 라벨**을 원본에 붙이기.
- `c.startswith(fp_prefixes)` : 열 이름이 지문 접두사로 시작하면 지문 열(튜플을 주면 그중 하나라도 매칭). `all_cols`=지문+descriptor 전체.
- `def xy(cols, part):` : 원하는 특징 열(cols)과 split(part=train/val/test)의 X, y를 꺼내는 함수. `df.loc[마스크, 열]`=조건 행+열 선택, `.to_numpy()`=배열로.

### 셀 2 — 후보 알고리즘 5종 + 지표 함수
각 모델을 (결측 채움 + 필요시 스케일 + 모델) 파이프라인으로 정의하고, 6가지 지표 계산 함수를 만든다.

In [ ]:
# 후보 알고리즘 5종 (트리 4 + 선형 1). 선형은 스케일 필요 → Pipeline으로 묶음
def candidates():
    imp = lambda: SimpleImputer(strategy="median")     # 3D desc의 NaN(대형분자) 채움
    return {
        "RandomForest": Pipeline([("i", imp()), ("m", RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1))]),
        "ExtraTrees": Pipeline([("i", imp()), ("m", ExtraTreesClassifier(
            n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1))]),
        "LightGBM": Pipeline([("i", imp()), ("m", LGBMClassifier(
            n_estimators=400, class_weight="balanced", random_state=42, n_jobs=-1, verbosity=-1))]),
        "XGBoost": Pipeline([("i", imp()), ("m", XGBClassifier(
            n_estimators=400, random_state=42, n_jobs=-1, eval_metric="logloss", verbosity=0))]),
        "LogReg": Pipeline([("i", imp()), ("s", StandardScaler()), ("m", LogisticRegression(
            max_iter=2000, class_weight="balanced"))]),
    }

def metrics(y, proba):
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return dict(MCC=matthews_corrcoef(y, pred), ROC_AUC=roc_auc_score(y, proba),
                PR_AUC=average_precision_score(y, proba),
                Accuracy=(tp+tn)/(tp+tn+fp+fn),
                Recall=tp/(tp+fn) if (tp+fn) else 0, Precision=tp/(tp+fp) if (tp+fp) else 0)

🔎 **코드 뜯어보기 (셀 2)**
- `def candidates():` : 이름→파이프라인 딕셔너리 반환. `Pipeline([("i", SimpleImputer(...)), ("m", 모델)])`=결측 채움 후 학습. LogReg만 `StandardScaler` 추가(선형은 스케일 필요, 트리는 불필요).
- `n_estimators`·`class_weight="balanced"` 등이 **하이퍼파라미터**. `imp = lambda: SimpleImputer(...)`=매번 새 임퓨터를 만드는 짧은 함수.
- `def metrics(y, proba):` : 확률→0/1 후 혼동행렬로 MCC·ROC-AUC·PR-AUC·Accuracy·Recall·Precision 계산(20에서 설명한 것과 동일).

### 셀 3 — 비교→상위4→앙상블→시험 평가 (전체특징 & 지문전용)
한 특징집합에 대해 5모델을 학습·검증채점하고, 상위 4를 골라 앙상블해 시험셋 성능을 낸다. 이를 (A)전체특징·(B)지문전용 두 번 수행.

In [ ]:
# 한 특징집합에 대해: 5모델 학습→검증셋 채점→상위4 선정→앙상블→시험셋 평가
def run_pipeline(cols, tag):
    Xtr, ytr = xy(cols, "train"); Xva, yva = xy(cols, "val"); Xte, yte = xy(cols, "test")
    rows = {}
    fitted = {}
    for name, model in candidates().items():
        model.fit(Xtr, ytr)
        pv = model.predict_proba(Xva)[:, 1]
        rows[name] = metrics(yva, pv)
        fitted[name] = model
    tbl = pd.DataFrame(rows).T.sort_values("MCC", ascending=False)
    print(f"\n[{tag}] 검증셋 성능 (MCC 내림차순)")
    print(tbl[["MCC", "ROC_AUC", "PR_AUC", "Accuracy", "Recall", "Precision"]].round(3).to_string())
    top4 = tbl.index[:4].tolist()
    print(f"  → 상위 4 모델: {top4}")

    # 상위4로 소프트보팅 앙상블 → train으로 학습 → test 평가
    ens = VotingClassifier([(n, candidates()[n]) for n in top4], voting="soft", n_jobs=-1)
    ens.fit(Xtr, ytr)
    pte = ens.predict_proba(Xte)[:, 1]
    mt = metrics(yte, pte)
    print(f"  [{tag}] 앙상블 시험셋: MCC {mt['MCC']:.3f} | ROC-AUC {mt['ROC_AUC']:.3f} | "
          f"PR-AUC {mt['PR_AUC']:.3f} | Acc {mt['Accuracy']:.3f} | Recall {mt['Recall']:.3f} | Prec {mt['Precision']:.3f}")
    return tbl, top4, ens, mt

# (A) 전체 특징 = 최고 성능 평가 / (B) fingerprint 전용 = 스크리닝 배포용
tblA, top4A, ensA, mtA = run_pipeline(all_cols, "전체특징(fp+2D+3D)")
tblB, top4B, ensB, mtB = run_pipeline(fp_cols, "fingerprint전용(배포용)")

🔎 **코드 뜯어보기 (셀 3)**
- `model.fit(Xtr, ytr)` : **훈련셋으로 학습**. `model.predict_proba(Xva)[:,1]` : **검증셋** 확률 예측(모델 고르는 기준).
- `pd.DataFrame(rows).T.sort_values("MCC", ascending=False)` : 모델별 지표표를 MCC 내림차순 정렬. `tbl.index[:4]`=상위 4 모델 이름.
- `VotingClassifier([(name, pipe) ...], voting="soft")` : 상위 4 모델의 **확률을 평균**하는 앙상블. `.fit(Xtr,ytr)` 후 `predict_proba(Xte)`로 **시험셋**에서 최종 평가(딱 한 번).
- (A) `all_cols`(전체) / (B) `fp_cols`(지문만) 두 번 실행 — A는 최고 성능, B는 배포용.

### 셀 4 — 배포용 앙상블 저장
스크리닝(08)에서 쓸 **fingerprint 전용 앙상블**과 모델 비교표를 저장한다.

In [ ]:
# 배포용(fingerprint 전용) 앙상블 저장 → 08 스크리닝에서 사용
with open("data/HSD17B13_ensemble_screen_model.pkl", "wb") as f:
    pickle.dump({"ensemble": ensB, "fp_cols": fp_cols, "fp_prefixes": list(fp_prefixes),
                 "nbits": 1024, "top4": top4B}, f)
# 비교표도 저장
tblA.assign(feature="all").reset_index().rename(columns={"index":"model"}).to_csv(
    "data/HSD17B13_model_compare_1to1.csv", index=False)
print("\n저장: data/HSD17B13_ensemble_screen_model.pkl (fingerprint 앙상블) + 비교표 CSV")
print(f"요약 → 전체특징 앙상블 test MCC {mtA['MCC']:.3f} / fingerprint 앙상블 test MCC {mtB['MCC']:.3f}")

🔎 **코드 뜯어보기 (셀 4)**
- `pickle.dump({"ensemble": ensB, "fp_cols": fp_cols, ...}, f)` : 지문 앙상블과 **지문 열 순서**를 함께 저장(08이 같은 순서로 지문을 만들어 예측해야 하므로).
- 전체특징(A) 앙상블은 '최고 성능 참고용', 배포(08)는 (B) 지문 앙상블 사용.